### 02. Label AI passages with an LLM judge

**Goal:** pick 30 random rules, cut out the passages that mention AI, and have Gemini label each rule using a codebook. Team members label the same 30 rules by hand to check Gemini's labels.

**Opens:** `fr_rules.csv` (from notebook 01)

**Saves:** `hand_labels.csv` (passages + hand labels), `gemini_labels.csv`

**AI assistance:** The code in this notebook was drafted alongside Claude and was reviewed, run, and checked by our team.

Links to official documentation are in the last cell.

#### Step 1: Import libraries

In [2]:
import os        # checks if a file is already saved
import re        # finds words in text, like "artificial intelligence"
import json      # reads Gemini's replies
import time      # pauses the code between Gemini requests

import pandas as pd               
import requests                   
from bs4 import BeautifulSoup    # removes web-page code (HTML) from downloaded text

#### Step 2: Collating full Data
After experimenting with the Gemini API and labelling, we're ready to use the codebook to label all of our texts.

In [ ]:
# Open the table of rules saved by notebook 01
rules = pd.read_csv("fr_rules.csv")

In [ ]:
# A function that downloads one rule's full text and cleans it up
def download_text(url):
    """Download a rule's full text and return it as plain text."""
    response = requests.get(url, timeout=60)      # send the request and wait up to 60 seconds
    response.raise_for_status()                   # stop with an error if the request failed
    text = BeautifulSoup(response.text, "html.parser").get_text()   # remove web-page code (HTML)
    text = " ".join(text.split())            # replace line breaks and extra spaces with single spaces
    return text               # show the first 500 characters

In [ ]:
# The search terms, joined with | ("or")
search_pattern = "artificial intelligence|machine learning|automated decision|deep learning|large language model"

# A function that finds up to 3 passages around the search terms
def get_passages(text):
    """Return up to 3 passages around the search terms, joined into one piece of text."""
    passages = []        # an empty list for the passages
    last_end = 0         # where the previous passage ended

    # Go through each place a search term appears (ignoring capital letters)
    for match in re.finditer(search_pattern, text, flags=re.IGNORECASE):
        if match.start() < last_end:              # this mention is already inside the previous passage
            continue                              # skip it and go to the next mention

        start = max(match.start() - 500, 0)       # 500 characters before (but not before the start of the text)
        end = match.end() + 500                   # 500 characters after
        passages.append(text[start:end])          # cut out the passage and add it to the list
        last_end = end

        if len(passages) == 3:               # stop after 3 passages
            break

    return "\n\n...\n\n".join(passages)      # join the passages with "..." between them


In [ ]:
# An empty list for each rule's passages
all_passages = []

# Repeat for each rule in the sample
for url in rules["raw_text_url"]:
    text = download_text(url)                  # download the full text
    all_passages.append(get_passages(text))    # find the passages and add them to the list

# Add the passages to the sample table as a new column
rules["passages"] = all_passages

# Check: rules where no passage was found (should be 0)
no_passages = rules[rules["passages"] == ""]
print("Rules with no passages found:", len(no_passages))

In [ ]:
#saving as csv file - this is all the rules with all the passages found in the text
rules.to_csv("fr_rules_with_passages.csv", index=False)

#### Step 3: Connect to Gemini

In [ ]:
from dotenv import load_dotenv     # reads the .env file
from google import genai           # connects to Gemini
from google.genai import types     # settings for Gemini

# Read the key from the .env file
load_dotenv()

# The Gemini model to use (the same one as in the Week 7 slides)
gemini_model = "gemini-3.5-flash-lite"

# If Gemini is busy (error 429 or 503), wait and try again, up to 3 times
retry = types.HttpRetryOptions(
    attempts=3,                    # try up to 3 times
    initial_delay=1.0,             # wait 1 second before the first retry
    exp_base=2,                    # double the wait each time (1 second, then 2, then 4)
    http_status_codes=[429, 503],  # the errors that mean "busy, try again"
)

# Connect to Gemini
client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"],
    http_options=types.HttpOptions(retry_options=retry),
)

print("Connected to Gemini")

#### Step 4: Gemini Labelling
Utilizing the same notebook from the second step, we will now apply it to the entire list of passages:


In [ ]:
# The instructions sent to Gemini with every rule (the same codebook as the table above)
codebook = """You are labeling passages from U.S. Federal Register rules for a research project.
Each passage mentions artificial intelligence (AI) or a related term.

Decide how the agency treats AI. Give one label:

regulate: The rule adds limits, oversight, or protections for AI (for example: addressing risks, harms, bias, safety, or privacy, or setting rules for how AI may be used).
promote: The rule encourages more use of AI (for example: funding AI, or pointing to its benefits, efficiency, or innovation) or demonstrates how federal agencies are already using or plans to use AI in the future.
:passing mention: The rule mentions AI without imposing or lifting government restrictions and passing any positive / negative use of age (for example: listing AI among other technologies or acknowledging a public comment regarding AI). 

Rules for tricky cases:
- Requiring that AI be tested, disclosed, or overseen counts as regulate, even if the rule allows AI.
- If a public commenter asks for something, label the agency's response, not the commenter's request.

Judge only from the text given. Also give a two to three sentence reason.
"""

# The format Gemini's answer must follow: one of the 5 labels, plus a reason
answer_format = {
    "type": "object",
    "properties": {
        "label": {"type": "string", "enum": ["regulate", "promote", "passing mention"]},
        "reason": {"type": "string"},
    },
    "required": ["label", "reason"],
}

# Gemini's settings: the codebook, the answer format, and temperature 0 for consistent answers
settings = types.GenerateContentConfig(
    system_instruction=codebook,
    response_mime_type="application/json",    # answer in JSON format
    response_schema=answer_format,
    temperature=0,                            # 0 = give the same answer each time
)

In [ ]:
# A function that sends one rule to Gemini and returns its label and reason
def ask_gemini(title, passages):
    """Send one rule's title and passages to Gemini. Return (label, reason)."""
    text = "Title: " + title + "\n\nPassages:\n" + passages # the text Gemini reads: the title, then the passages

    # Send the text with the settings from Step 6 (codebook, answer format, temperature)
    response = client.models.generate_content(
        model=gemini_model,
        contents=text,
        config=settings,
    )

    answer = json.loads(response.text) # Turn Gemini's JSON answer into a Python dictionary
    return answer["label"], answer["reason"]

In [ ]:
# An empty list for Gemini's answers
results = []

# Repeat for each of the 30 rules
for i in range(len(rules)):                      
    document_number = rules.loc[i, "document_number"]
    title = rules.loc[i, "title"]
    passages = rules.loc[i, "passages"]

    # Ask Gemini. If something goes wrong, record "error" instead of stopping.
    try:
        label, reason = ask_gemini(title, passages)
    except Exception as problem:
        label = "error"
        reason = str(problem)                     # save the error message as the reason

    # Add this rule's answer to the list
    results.append({
        "document_number": document_number,
        "gemini_label": label,
        "gemini_reason": reason,
    })

    print(i + 1, "of", len(rules), ":", label)   # show progress
    time.sleep(8)                                 # wait 8 seconds before the next request


In [ ]:

# Turn the list into a table and save it
fr_rules_all_gemini_labels = pd.DataFrame(results)
fr_rules_all_gemini_labels.to_csv("fr_rules_all_gemini_labels.csv", index=False)
print("Saved fr_rules_all_gemini_labels.csv")


In [ ]:

# Count how many rules got each label
fr_rules_all_gemini_labels["gemini_label"].value_counts()

#### Joining labels to metadata and writing to CSV for analysis

In [11]:
gemini_labels = pd.read_csv("../data/fr_rules_all_gemini_labels.csv")
rules_metadata = pd.read_csv("../data/fr_rules_with_passages.csv")
fr_merge_total = pd.merge(gemini_labels, rules_metadata, on="document_number", how="left")
fr_rules_labels_metadata = fr_merge_total[['document_number', 'gemini_label', 'gemini_reason', 'title', 'publication_date', 'matched_terms', 'agency', 'administration', 'passages']]
fr_rules_labels_metadata.to_csv("../data/fr_rules_labels_metadata.csv", index=False)

#### Documentation

* **Gemini API:** [Python library (google-genai)](https://googleapis.github.io/python-genai/) · [structured output (answer format)](https://ai.google.dev/gemini-api/docs/structured-output) · [system instructions (codebook)](https://ai.google.dev/gemini-api/docs/text-generation#system-instructions)
* **python-dotenv:** [reading the .env file](https://pypi.org/project/python-dotenv/)
* **requests:** [quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/)
* **BeautifulSoup:** [documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
* **pandas:** [read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) · [sample](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html) · [reset_index](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html) · [copy](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html) · [to_csv](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html)
* **Python:** [re.finditer (finding words in text)](https://docs.python.org/3/library/re.html#re.finditer) · [os.path.exists](https://docs.python.org/3/library/os.path.html#os.path.exists) · [json.loads](https://docs.python.org/3/library/json.html#json.loads) · [try / except](https://docs.python.org/3/tutorial/errors.html#handling-exceptions)